# 日本储能前瞻研究 v2.2：完整复算

三页研究问题对应独立数据与计算。实绩用于历史解释，不证明因果；项目为公开样本；2030数字为C压力情景。此Notebook实际执行全部22,240次DP调度与37,530次新预留LP，计算通常需要数分钟。原基准12,510个LP日沿用受保护资产。

## 1 原始证据与保护资产

固定取回快照和SHA256。新计算在单独临时目录完成，不覆盖正式数据。

In [1]:
from pathlib import Path
import json,hashlib,sys,shutil,subprocess,uuid,time
import numpy as np
ROOT=Path.cwd().resolve()
assert (ROOT/'research/forward/sources.json').exists()
sys.path.insert(0,str(ROOT/'scripts'))
read=lambda p:json.loads((ROOT/p).read_text(encoding='utf-8'))
baseline=read('research/forward/baseline.json')
for key,sha in baseline['protected_files'].items():
 assert hashlib.sha256((ROOT/key).read_bytes()).hexdigest()==sha,key
sources=read('research/forward/sources.json')
for s in sources:
 if s['status']=='downloaded':assert hashlib.sha256((ROOT/s['raw_file']).read_bytes()).hexdigest()==s['sha256'],s['id']
SCRATCH=ROOT/'tmp'/('forward-notebook-replay-'+uuid.uuid4().hex[:10])
assert SCRATCH.resolve().is_relative_to(ROOT)
(SCRATCH/'research/forward').mkdir(parents=True)
shutil.copytree(ROOT/'research/forward/raw',SCRATCH/'research/forward/raw')
for key in ['research/forward/sources.json','research/forward/scenario-definitions.json','research/jepx-normalized.csv','research/p1/projects.json','dist/data/lab-v2.json']:
 target=SCRATCH/key;target.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(ROOT/key,target)
import forward_drivers as fd, forward_projects as fp, forward_driver_windows as fw
fd.ROOT=SCRATCH;fd.OUT=SCRATCH/'research/forward';fp.ROOT=SCRATCH;fp.OUT=fd.OUT;fw.ROOT=SCRATCH;fw.OUT=fd.OUT
(SCRATCH/'dist/data').mkdir(parents=True,exist_ok=True)
def run(command,timeout=900):
 result=subprocess.run(command,cwd=ROOT,text=True,encoding='utf-8',capture_output=True,timeout=timeout)
 assert result.returncode==0,(command,result.stdout,result.stderr)
 print(result.stdout[-700:])
 return result
print('Protected files',len(baseline['protected_files']),'official raw snapshots',sum(s['status']=='downloaded' for s in sources))
print('Python',sys.version.split()[0],'NumPy',np.__version__,'isolated replay',SCRATCH.name)


Protected files 39 official raw snapshots 58
Python 3.12.10 NumPy 2.5.1 isolated replay forward-notebook-replay-f0ba8650f8


## 2 JST、频率、单位与联络线符号

九州CSV正值送出，转为净受电时反号。两槽MW与价格均值形成小时；速報差额保留。

In [2]:
records,hours,quality=fd.normalize()
assert len(records)==61248 and len(hours)==30624
assert len({(r['region'],r['date_jst'],r['slot']) for r in records})==len(records)
assert all(r['start_jst'].startswith(r['date_jst']) and r['start_jst'].endswith('+09:00') for r in records)
assert quality['duplicates']==0 and quality['excluded']==[]
assert all(r['component_sum_to_total_diff_mw']==0 for r in records)
assert all(r['interchange_net_import_mw']==r['source_interchange_mw']*(-1 if r['region']=='kyushu' else 1) for r in records)
assert hashlib.sha256((fd.OUT/'drivers-normalized-30min.csv').read_bytes()).hexdigest()==read('research/forward/drivers-quality.json')['normalized_sha256']
print(json.dumps(quality['regions'],ensure_ascii=False))


{"chubu": {"expected_30min_rows": 30624, "actual_30min_rows": 30624, "hourly_rows": 15312, "maximum_supply_demand_difference_mw": 0.0, "maximum_component_to_total_difference_mw": 0.0, "supply_demand_residual_rows": 0, "source_status_rows": {"confirmed_label": 0, "preliminary_or_unlabelled": 30624}}, "kyushu": {"expected_30min_rows": 30624, "actual_30min_rows": 30624, "hourly_rows": 15312, "maximum_supply_demand_difference_mw": 26.0, "maximum_component_to_total_difference_mw": 0.0, "supply_demand_residual_rows": 1298, "source_status_rows": {"confirmed_label": 29184, "preliminary_or_unlabelled": 1440}}}


## 3 供需解释模型

2025训练，2026年1–9月留出；train-only标准化、固定α=100和日块置换seed=20261007。

In [3]:
regenerated=fd.analyze(records,hours,quality)
fw.main()
regenerated=json.loads((fd.OUT/'drivers-analysis.json').read_text(encoding='utf-8'))
saved=read('research/forward/drivers-analysis.json')
for region,v in regenerated['regions'].items():
 for key in ['mae','rmse','r2']:
  assert abs(v['holdout']['full'][key]-saved['regions'][region]['holdout']['full'][key])<1e-9
 assert v['train_dates'][1]<v['test_dates'][0]
 for a,b in zip(v['importance'],saved['regions'][region]['importance']):
  assert a['feature']==b['feature']
  assert abs(a['mae_increase']-b['mae_increase'])<1e-9
 print(region,'holdout MAE',v['holdout']['full']['mae'],'calendar MAE',v['holdout']['calendar_only']['mae'],'first driver',v['importance'][0]['title'])
print(saved['statement'],saved['actual_inputs_notice'])


{"chubu": {"train": 8760, "test": 6552, "mae": 6.907722071049387, "calendar_mae": 6.581371395374653, "top": "负荷"}, "kyushu": {"train": 8760, "test": 6552, "mae": 3.2732240080713124, "calendar_mae": 4.036945578279217, "top": "光伏出力/负荷"}}
2 regions × 3 descriptive windows; frozen holdout model unchanged
chubu holdout MAE 6.907722071049387 calendar MAE 6.581371395374653 first driver 负荷
kyushu holdout MAE 3.2732240080713124 calendar MAE 4.036945578279217 first driver 光伏出力/负荷
这些结果用于解释历史共变关系，不证明因果关系。 负荷、出力、限发和联络线使用事后实绩；不能提前用于预测。


## 4 项目解析与去重

项目自己所在的官方表格核验容量与日期。共用SPV不等于同一站，重复公告不增加站数。

In [4]:
fp.main()
projects=json.loads((fd.OUT/'storage-projects.json').read_text(encoding='utf-8'))
assert projects==read('research/forward/storage-projects.json')
observations=json.loads((fd.OUT/'project-observations.json').read_text(encoding='utf-8'))
dedup,audit=fp.deduplicate(observations)
assert dedup==projects['projects'] and fp.aggregate(dedup)==projects['aggregate']
assert len(observations)==48 and len(dedup)==22
assert projects['taoke_comparison']['projects']==9
print('48 official observations ->22 physical stations; 19 future,',projects['aggregate']['future']['mw'],'MW')


{"sample": {"projects": 22, "mw": 1225.9, "mwh_disclosed_sum": 3451.3, "mwh_disclosed_projects": 18, "mwh_unknown_projects": 4}, "future": {"projects": 19, "mw": 1160.9, "mwh_disclosed_sum": 3283.8, "mwh_disclosed_projects": 15, "mwh_unknown_projects": 4}, "dedup": {"observations": 48, "physical_projects": 22, "merged_duplicate_observations": 26, "entity_resolution": "curated explicit canonical asset IDs with source-level names/locations/ratings; no fuzzy merge by sharedSPV"}}
48 official observations ->22 physical stations; 19 future, 1160.9 MW


## 5 全量现货压力调度

预测变换只使用自身中位数；重新生成动作后按压力实际价结算。全部8个价格形状×5区域×278天×2策略。

In [5]:
run(['node','scripts/prepare_forward_spot.mjs','--output-directory',str(fd.OUT),'--force'])
definitions=read('research/forward/scenario-definitions.json')
count=0
for definition in definitions['prices']:
 fresh=json.loads((fd.OUT/'spot-cases'/f"{definition['id']}.json").read_text(encoding='utf-8'))
 original=read(f"research/forward/spot-cases/{definition['id']}.json")
 assert fresh['inputs']==original['inputs']
 for region,strategies in fresh['series'].items():
  for strategy,rows in strategies.items():
   for a,b in zip(rows,original['series'][region][strategy]):
    assert a['date']==b['date']
    for k in ['grossRevenue','chargingCost','dischargeRevenue','chargeMWh','dischargeMWh']:assert abs(a[k]-b[k])<1e-5
    if strategy=='ridge':assert a['informationCutoff']<a['date'] and a['trainingEnd']<a['date']
    count+=1
assert count==22240
print('FULL spot replay',count,'schedules /',count*48,'slots; all daily totals reproduced')


edules; physical constraints passed
renewable tokyo: 278 days × 2 schedules; physical constraints passed
renewable kansai: 278 days × 2 schedules; physical constraints passed
renewable kyushu: 278 days × 2 schedules; physical constraints passed
renewable tohoku: 278 days × 2 schedules; physical constraints passed
combined chubu: 278 days × 2 schedules; physical constraints passed
combined tokyo: 278 days × 2 schedules; physical constraints passed
combined kansai: 278 days × 2 schedules; physical constraints passed
combined kyushu: 278 days × 2 schedules; physical constraints passed
combined tohoku: 278 days × 2 schedules; physical constraints passed
{"status":"passed","new_schedules":22240}

FULL spot replay 22240 schedules / 1067520 slots; all daily totals reproduced


## 6 全量预留功率LP

相同2MW/8.14MWh电池；2−r现货MW，SOC首尾50%、5分钟双向余量，9档预留。

In [6]:
run([sys.executable,'-X','utf8','scripts/prepare_forward_frontier.py','--output-directory',str(fd.OUT),'--force'])
solves=0
for definition in definitions['prices']:
 if not definition['frontier'] or definition['id']=='base':continue
 for region in ['chubu','tokyo','kansai','kyushu','tohoku']:
  key=f"frontier-cases/{definition['id']}-{region}.json"
  fresh=json.loads((fd.OUT/key).read_text(encoding='utf-8'));original=read('research/forward/'+key)
  assert fresh['inputs']==original['inputs']
  for a,b in zip(fresh['rows'],original['rows']):
   assert a['reserveMW']==b['reserveMW']
   for x,z in zip(a['daily'],b['daily']):
    for k in ['grossRevenue','chargingCost','dischargeRevenue','chargeMWh','dischargeMWh']:assert abs(x[k]-z[k])<1e-5
  solves+=fresh['receipt']['lp_solves']
assert solves==37530
print('FULL reservation replay',solves,'LP solves; 5 areas,3 new price shapes,9 reserve levels,278 days')


 "region": "tokyo", "lp_solves": 2502, "passed": true}
{"case": "renewable", "region": "kansai", "lp_solves": 2502, "passed": true}
{"case": "renewable", "region": "kyushu", "lp_solves": 2502, "passed": true}
{"case": "renewable", "region": "tohoku", "lp_solves": 2502, "passed": true}
{"case": "combined", "region": "chubu", "lp_solves": 2502, "passed": true}
{"case": "combined", "region": "tokyo", "lp_solves": 2502, "passed": true}
{"case": "combined", "region": "kansai", "lp_solves": 2502, "passed": true}
{"case": "combined", "region": "kyushu", "lp_solves": 2502, "passed": true}
{"case": "combined", "region": "tohoku", "lp_solves": 2502, "passed": true}
Named-case frontier reruns complete

FULL reservation replay 37530 LP solves; 5 areas,3 new price shapes,9 reserve levels,278 days


## 7 成本、容量价值与经营贡献

沿用v2.1模型；容量价格不是bid cap，成本情景不是实际账单。

In [7]:
run(['node','scripts/prepare_forward_payload.mjs','--output-directory',str(fd.OUT)])
payload=json.loads((fd.OUT/'scenarios-output.json').read_text(encoding='utf-8'))
saved=read('research/forward/scenarios-output.json')
for scenario,regions in payload['scenarioResults'].items():
 for region,v in regions.items():
  for kind in ['perfect','ridge','combined']:
   for key in ['gross','cost','net']:assert abs(v[kind][key]-saved['scenarioResults'][scenario][region][kind][key])<1e-5
flat=json.loads((fd.OUT/'scenario-default-results.json').read_text(encoding='utf-8'))
assert len(flat)==540
for a,b in zip(flat,read('research/forward/scenario-default-results.json')):
 for key in ['combined_contribution','capacity_gross','lost_spot_gross']:
  assert abs(a[key]-b[key])<1e-5
print('540 scenario×area×cost×reserve results and protected baseline reproduced')


50580596923828e-9,"negative_forecast_estimates_preserved":true,"forecast_transform_uses_own_median":true,"capacity_values_are_C_not_bid_caps":true,"identical_resource_constraints":true},"chubu":{"base":{"ridge":8913639.074098397,"combined":21240880.371939432,"retained":1,"halfPrice":0.42219822863092943},"storage":{"ridge":4553316.982687488,"combined":14064787.952709662,"retained":0.6621565446642292,"halfPrice":1.4139851438509243},"renewable":{"ridge":15760338.958104612,"combined":22906928.624770418,"retained":1.0784359322051427,"halfPrice":-0.28218634132387255},"combined":{"ridge":9483069.495065216,"combined":13152086.534312934,"retained":0.6191874491081681,"halfPrice":1.1855167651280065}}}

540 scenario×area×cost×reserve results and protected baseline reproduced


## 8 报告图表与最终保护核对

原始资产→计算→网站→报告所有数量可以追溯。透明压力测试不写成确定收益预测。

In [8]:
run(['node','scripts/prepare_forward_report_data.mjs','--research-directory',str(fd.OUT)])
figures=json.loads((fd.OUT/'report-figures.json').read_text(encoding='utf-8'))
for a,b in zip(figures,read('research/forward/report-figures.json')):
 assert a['id']==b['id'] and a['labels']==b['labels'] and a['unit']==b['unit']
 for x,z in zip(a['series'],b['series']):np.testing.assert_allclose(x['values'],z['values'],rtol=1e-9,atol=1e-9)
assert len(figures)==11
for key,sha in baseline['protected_files'].items():assert hashlib.sha256((ROOT/key).read_bytes()).hexdigest()==sha,key
print('11 report figure datasets reproduced; all39 previous assets still byte-identical')
print('REPLAY_PASSED: raw ->JST/MW ->attribution/projects ->C inputs ->full DP/LP ->costs ->site ->report figures')


11 chart contracts; 16-page technical report data

11 report figure datasets reproduced; all39 previous assets still byte-identical
REPLAY_PASSED: raw ->JST/MW ->attribution/projects ->C inputs ->full DP/LP ->costs ->site ->report figures
